<img src="./figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2024 (Burgas, Bulgaria), On-Site Round](https://ioai-official.org/bulgaria-2024)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2024/blob/main/On-Site-Round/Help_BOBAI/Help_BOBAI.ipynb)

# Help BOBAI: More classification in an unknown language

## Portable execution layer

This tracked copy preserves the latest local work and its saved evidence. Downloads are verified against `manifest.json` and stored only in this task's ignored `.data/` and `.cache/` directories. Generated files go to `outputs/`. Set `PORTABLE_IOAI_SMOKE=1` to execute only the small CPU portability contract; the original workload cells are tagged `full-run`.


In [ ]:
from pathlib import Path
import json
import os
import sys

def _find_portable_root():
    start = Path.cwd().resolve()
    for parent in (start, *start.parents):
        for candidate in (parent, parent / 'olympiads' / 'portable_ioai'):
            if (candidate / 'bootstrap.py').is_file() and (candidate / 'manifest.json').is_file():
                return candidate
    raise FileNotFoundError(
        'Could not locate olympiads/portable_ioai. Start Jupyter from the '
        'repository root or this notebook directory, after running setup.ps1.'
    )

PORTABLE_ROOT = _find_portable_root()
if str(PORTABLE_ROOT) not in sys.path:
    sys.path.insert(0, str(PORTABLE_ROOT))
from bootstrap import NotebookContext, load_hf_datasets, smoke_task

PORTABLE = NotebookContext('help_bobai').prepare_paths()
DATA_DIR = PORTABLE.data_dir
OUTPUT_DIR = PORTABLE.output_dir
SMOKE_MODE = os.environ.get('PORTABLE_IOAI_SMOKE', '').lower() in {'1', 'true', 'yes'}
os.environ[PORTABLE.spec['data_environment']] = str(DATA_DIR)
if 'help_bobai' == 'home_task_3':
    model_mode = 'smoke' if SMOKE_MODE else 'full'
    model_spec = PORTABLE.spec['models'][model_mode]
    os.environ['PORTABLE_IOAI_HT3_MODEL'] = model_spec['id']
    os.environ['PORTABLE_IOAI_HT3_MODEL_REVISION'] = model_spec['revision']
print(json.dumps({**PORTABLE.describe(), 'smoke_mode': SMOKE_MODE}, indent=2))


In [ ]:
if SMOKE_MODE:
    SMOKE_RESULT = smoke_task('help_bobai', ensure=True)
    print(json.dumps(SMOKE_RESULT, indent=2, default=str))
else:
    print('Portable context ready; continuing with the preserved full-workload cells.')


<img src="./figs/Help BOBAI Fig 1.png" width="700">

## Background
Last time you heard from Bob, he asked you to help him by building a classifier for a new unknown language. The client, Amoira, was happy with your solution so Bob instructed his team to deploy the new model and after some heavy optimization and careful unit testing, the service was deployed and has been running smoothly since.

## Task

This very morning, Amoira returned with a request to extend the number of classes which the classifier can handle from 5 to 7. And this has to be done *today*!

Amoira has provided labeled data for the new classes. With more time, Bob could just use your earlier solution to train a new model on the union of the old and new data, right? The trouble is that the deployment of a new model is a complex process and cannot be done in a day, so the solution has to be built entirely around the model already deployed. Bob has once more come to you for help, as you know the task best.

Whatsmore, Amoira's security concerns have grown even further with the addition of the new data, so they have requested that Bob does not release the text in any form - what if someone managed to decrypt it! So Bob has provided you with a precomputed and cached encoding of all available data: the train and dev set previously used for the 5-way classification, and the new data Amoira provided for the 2 additional classes. The encoding is the output of the pooling layer in mBERT, so is fits right into the classifier previously trained.

Your task is to build a solution for 7-way classification, while operating within the following constraints:

*   The solution can use the 5-way classifier, but cannot change the parameters of the classifier or add any new learned parameters.

*   You are allowed to compute averages and distances between the data encodings.

*   The solution should be reproducible in under 1 hour on an L4 GPU card.

*   The classifier has to perform inference on any random 500 data samples in under 2 minutes on an L4 GPU card.

## Deliverables

You need to submit:

*   Working code that can be used to reproduce and test your best model.
  * In this Colab notebook.
  * Reproducing your best model means that starting from the baseline classifier, we should be able to arrive at your final best model by executing the cells of the notebook.
*   The predictions on the test data (released two hours before the end of the competition).

**You absolutely need to ensure that:**

(1) your notebook is executable from top to bottom

(2) that the notebook contains the full code needed to reproduce your model

(3) that it can run on an L4 GPU



## Training Dataset

In [345]:
def print_dir(obj):
    print(f"---| {type(obj)} |---| {obj.__class__} |---")
    for i in dir(obj):
        if i.startswith("_"):
            continue
        print(i)

In [346]:
import torch

DATA_DIR = PORTABLE.ensure_data()
dataset = torch.load(
    DATA_DIR / "training_set" / "train-dev_dataset_with_labels.pt",
    map_location="cpu",
    weights_only=True,
)
inputs = dataset[:, :, :-1]
labels = dataset[:, :, -1]


In [347]:
print(inputs.shape)
print(labels.dtype)
print(labels.max())
print(labels.min())
# uncertain_mask = torch.where(labels>4, labels, False)
uncertain_mask = (labels>4)
uncertain_inputs = inputs[uncertain_mask].reshape(-1, 1, 768)
uncertain_labels = labels[uncertain_mask] - 5
print(uncertain_inputs.shape)
print(uncertain_labels.shape)
print(uncertain_mask.dtype)

torch.Size([2473, 1, 768])
torch.float32
tensor(6.)
tensor(0.)
torch.Size([731, 1, 768])
torch.Size([731])
torch.bool


## Baseline Solution

Below you will find a very naive baseline solution: given an input vector, we use either randomly assign one of the new labels (5 and 6) with uniform probability over a 7-way classification, or we use the base classifier to make a prediction.

You can replace the code below with your solution.

In [348]:
model_inputs = inputs.reshape(-1,768).numpy()
model_labels = torch.where(labels<4, 0, labels-4).reshape(-1).numpy()
print(labels)
print(model_labels.max())

tensor([[3.],
        [2.],
        [6.],
        ...,
        [1.],
        [2.],
        [3.]])
2.0


In [349]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
model.fit(model_inputs, model_labels);
preds = model.predict(inputs.reshape(-1,768))
print(preds.max())
score = accuracy_score(model_labels, preds)
print(score)

2.0
0.819652244237768


In [350]:
import torch
import random

class SevenWayClassifier():
  def __init__(self, ):
    base_clf = torch.nn.Linear(in_features=768, out_features=5, bias=True)
    base_clf.load_state_dict(torch.load(
        DATA_DIR / "training_set" / "base_classifier.pth",
        map_location="cpu", weights_only=True,
    ))
    self.base_clf = base_clf

  def predictions(self, input_vector):
    with torch.no_grad():
      logits = self.base_clf(input_vector)
      preds = torch.softmax(logits, 1)
    return preds

  def base_classification(self, input_vector):
    predicted_class = self.predictions(input_vector).argmax(dim=1).numpy()[0]
    return int(predicted_class)
  

  def __call__(self, input_vector):
    pred = model.predict(input_vector)
    if pred>0:
      predicted_class = pred.item()+4
    else:
      predicted_class = self.base_classification(input_vector)

    return int(predicted_class)

clf = SevenWayClassifier()

## Inference and Evaluation

In [351]:
from sklearn.metrics import f1_score

def compute_f1(labels, predictions):
  return f1_score(np.asarray(labels).reshape(-1), np.asarray(predictions).reshape(-1), average='macro')

In [352]:
from tqdm import tqdm

def inference(clf, input_vectors):
  predictions = []
  for sample in tqdm(input_vectors):
    predictions.append(clf(sample))
  return predictions

In [353]:

predictions = inference(clf, inputs)


100%|██████████| 2473/2473 [00:18<00:00, 133.69it/s]


In [354]:
print(labels, predictions)
f1 = compute_f1(labels, predictions)
print('\nNaive solution F1', f1)

tensor([[3.],
        [2.],
        [6.],
        ...,
        [1.],
        [2.],
        [3.]]) [np.int64(3), np.int64(2), np.int64(3), np.int64(2), np.int64(2), np.int64(0), np.int64(3), np.int64(3), np.int64(4), np.int64(3), 6.0, np.int64(2), np.int64(1), np.int64(1), np.int64(4), np.int64(1), np.int64(3), 6.0, np.int64(2), np.int64(3), np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(0), np.int64(1), np.int64(3), np.int64(1), np.int64(4), np.int64(3), np.int64(1), 6.0, np.int64(2), np.int64(2), np.int64(2), np.int64(1), 5.0, np.int64(3), 5.0, np.int64(3), np.int64(2), np.int64(1), np.int64(3), 6.0, np.int64(1), np.int64(4), np.int64(3), np.int64(1), np.int64(3), np.int64(0), np.int64(0), np.int64(0), 5.0, 5.0, np.int64(1), np.int64(0), np.int64(2), np.int64(3), 6.0, np.int64(4), np.int64(1), np.int64(1), np.int64(4), np.int64(2), np.int64(3), 5.0, np.int64(3), np.int64(0), 5.0, np.int64(0), 6.0, 6.0, np.int64(3), np.int64(1), np.int64(4), 5.0, np.int64(4), np.int64(4),

## Validation Dataset

In [355]:
# The leaderboard may or may not work... If it doesn't forgive us. We will try to get it running.

import pandas as pd
import numpy as np

def submission_to_csv(pred: np.ndarray, output_fpath: str = "submission.csv"):
    pred = np.array(pred).flatten()
    data_size = pred.size
    df = pd.DataFrame({
        "ID": np.arange(data_size),
        "class": pred
    })

    df["class"] = df["class"].astype(int)
    df.to_csv(output_fpath, index=False)

eval_inputs = torch.load(
    DATA_DIR / "Solution" / "validation_set" / "eval_dataset.pt",
    map_location="cpu", weights_only=True,
)

eval_predictions = inference(clf, eval_inputs)

submission_to_csv(eval_predictions, OUTPUT_DIR / "submission.csv")

100%|██████████| 200/200 [00:01<00:00, 128.56it/s]


## Test Dataset

In [356]:
# The pinned official test tensor is downloaded by the shared bootstrap.
test_inputs = torch.load(
    DATA_DIR / "Solution" / "test_set" / "test_dataset.pt",
    map_location="cpu",
    weights_only=True,
)
test_predictions = inference(clf, test_inputs)
prediction_path = OUTPUT_DIR / "Team Name_predictions.txt"
prediction_path.write_text(
    "\n".join(str(int(prediction)) for prediction in test_predictions),
    encoding="utf-8",
)
print("Saved", prediction_path)


100%|██████████| 700/700 [00:05<00:00, 125.14it/s]
